# Cross-validation and released-model verification

This notebook reruns the repeated cross-validation analysis for the mobility and carrier-type classifiers, reproduces the two released models on all 512 training entries, and verifies them against the fixed artifacts in `models/`.

**Required core environment:** Python 3.9, XGBoost 2.1.4, scikit-learn 1.6.1, NumPy 2.0.2, pandas 2.3.1, and joblib 1.5.3.

Run the notebook from a clone of the OFETnow repository after installing the pinned environment with `pip install -r requirements.txt`. All paths are resolved from the repository root; no local absolute paths are required.

## 0. Environment check (mandatory)

In [ ]:
import sys

REQUIRED_VERSIONS = {
    "python": "3.9",
    "xgboost": "2.1.4",
    "scikit-learn": "1.6.1",
    "numpy": "2.0.2",
    "pandas": "2.3.1",
    "joblib": "1.5.3",
}

ENV_VERIFIED = False


def require_verified_env():
    if not ENV_VERIFIED:
        raise RuntimeError("Run the environment-check cell before continuing.")


import joblib
import numpy
import pandas
import sklearn
import xgboost

installed = {
    "python": f"{sys.version_info.major}.{sys.version_info.minor}",
    "xgboost": xgboost.__version__,
    "scikit-learn": sklearn.__version__,
    "numpy": numpy.__version__,
    "pandas": pandas.__version__,
    "joblib": joblib.__version__,
}

print(f"{'package':<14}{'required':<12}{'installed':<12}status")
print("-" * 48)
for package, required in REQUIRED_VERSIONS.items():
    status = "OK" if installed[package] == required else "MISMATCH"
    print(f"{package:<14}{required:<12}{installed[package]:<12}{status}")

mismatched = [p for p, version in REQUIRED_VERSIONS.items() if installed[p] != version]
if mismatched:
    raise RuntimeError(
        f"Version mismatch: {', '.join(mismatched)}.\n"
        f"Interpreter: {sys.executable}\n"
        "Create a Python 3.9 environment and run: pip install -r requirements.txt"
    )

ENV_VERIFIED = True
print("\nEnvironment verified.")

## 1. Repository paths and experiment settings

In [ ]:
require_verified_env()

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier


def find_repo_root():
    """Find the repository whether the kernel starts in the root or notebooks/."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "requirements.txt").is_file() and (candidate / "models").is_dir():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Start Jupyter from the cloned OFETnow repository."
    )


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
MODEL_DIR = REPO_ROOT / "models"
OUTPUT_DIR = REPO_ROOT / "model_training_outputs"
MODEL_OUTPUT_DIR = OUTPUT_DIR / "verified_models"

X_PATH = DATA_DIR / "training_features.csv"
Y_PATH = DATA_DIR / "training_targets.csv"

SEED_PATH = MODEL_DIR / "cv_random_seeds.txt"
SEEDS = [int(value) for value in SEED_PATH.read_text(encoding="utf-8").splitlines()]
if len(SEEDS) != 10 or len(SEEDS) != len(set(SEEDS)):
    raise ValueError(f"Expected 10 unique random seeds in {SEED_PATH}")

N_SPLITS = 10
MODEL_METADATA = json.loads(
    (MODEL_DIR / "model_metadata.json").read_text(encoding="utf-8")
)
release_random_states = {
    model_name: model_config["random_state"]
    for model_name, model_config in MODEL_METADATA["models"].items()
}
if len(set(release_random_states.values())) != 1:
    raise ValueError(
        f"Expected a single archived random state for the released classifiers; "
        f"found {release_random_states}"
    )
RELEASE_MODEL_RANDOM_STATE = next(iter(release_random_states.values()))

BEST_PARAMS_MOBILITY = {
    "colsample_bytree": 0.6943386448447411,
    "learning_rate": 0.08869121921442981,
    "max_depth": 8,
    "n_estimators": 139,
    "subsample": 0.6404672548436904,
}
BEST_PARAMS_TYPE = {
    "colsample_bytree": 0.5316261237102475,
    "learning_rate": 0.1807997402055997,
    "max_depth": 9,
    "n_estimators": 200,
    "subsample": 0.7792221958558263,
}

print(f"Repository root: {REPO_ROOT}")
print(f"CV seeds ({len(SEEDS)}): {SEEDS}")
print(f"Folds per seed: {N_SPLITS}")
print("Released-model settings loaded from model_metadata.json")
print(f"Cross-validation output directory: {OUTPUT_DIR}")
print(f"Verified-model output directory: {MODEL_OUTPUT_DIR}")

## 2. Feature lists from the released repository

The notebook reads the released feature-name files directly. This avoids position-based indexing and verifies that the 2,213-column training matrix has exactly the expected names and order.

In [ ]:
require_verified_env()


def read_feature_names(path):
    names = path.read_text(encoding="utf-8").splitlines()
    if len(names) != len(set(names)):
        raise ValueError(f"Duplicate feature names in {path}")
    return names


ALL_FEATURES = read_feature_names(MODEL_DIR / "all_features.txt")
MOBILITY_FEATURES = read_feature_names(MODEL_DIR / "mobility_selected_features.txt")
TYPE_FEATURES = read_feature_names(MODEL_DIR / "carrier_type_selected_features.txt")

assert len(ALL_FEATURES) == 2213
assert len(MOBILITY_FEATURES) == 244
assert len(TYPE_FEATURES) == 334
assert set(MOBILITY_FEATURES).issubset(ALL_FEATURES)
assert set(TYPE_FEATURES).issubset(ALL_FEATURES)

print(f"All features: {len(ALL_FEATURES)}")
print(f"Mobility-classifier features: {len(MOBILITY_FEATURES)}")
print(f"Carrier-type classifier features: {len(TYPE_FEATURES)}")

## 3. Load and validate the training data

In [ ]:
require_verified_env()


def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


X_full = pd.read_csv(X_PATH)
y_full = pd.read_csv(Y_PATH)

assert X_full.shape == (512, 2213)
assert y_full.shape == (512, 2)
assert X_full.columns.tolist() == ALL_FEATURES
assert y_full.columns.tolist() == ["labels1", "labels2"]
assert not X_full.isna().any().any()
assert not y_full.isna().any().any()

X_mob = X_full.loc[:, MOBILITY_FEATURES]
y_mob = y_full["labels1"]
X_type = X_full.loc[:, TYPE_FEATURES]
y_type = y_full["labels2"]

print(f"Training-feature SHA-256: {sha256(X_PATH)}")
print(f"Training-target SHA-256:  {sha256(Y_PATH)}")
print(f"Mobility: {X_mob.shape}, labels: {y_mob.value_counts().sort_index().to_dict()}")
print(f"Carrier-type classifier: {X_type.shape}, labels: {y_type.value_counts().sort_index().to_dict()}")

---
## Part 1. Ten-seed, ten-fold stratified cross-validation

### 1.1 Run repeated stratified cross-validation

The ten archived random states define the repeated cross-validation protocol; this section does not rank the random states or select a released model. For the mobility classifier, precision, recall, and F1 refer to the high-mobility class (`labels1 = 1`). For carrier type, the notebook reports overall accuracy, one-vs-rest accuracy for each class, per-class precision/recall/F1, and macro-averaged precision/recall/F1. Carrier-type abbreviations are P for p-type, N for n-type, and Amb for ambipolar.

In [ ]:
require_verified_env()

mob_metrics, type_metrics = [], []
label_names = {0: "P", 1: "N", 2: "Amb"}

for seed_index, seed in enumerate(SEEDS):
    print(f"\n========== Seed {seed} ({seed_index + 1}/{len(SEEDS)}) ==========")

    mobility_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    mobility_true, mobility_pred = [], []
    for train_index, test_index in mobility_cv.split(X_mob, y_mob):
        classifier = XGBClassifier(
            eval_metric="logloss", random_state=seed, **BEST_PARAMS_MOBILITY
        )
        classifier.fit(X_mob.iloc[train_index], y_mob.iloc[train_index])
        mobility_true.extend(y_mob.iloc[test_index].to_numpy())
        mobility_pred.extend(classifier.predict(X_mob.iloc[test_index]))

    mob_row = {
        "seed": seed,
        "acc": accuracy_score(mobility_true, mobility_pred),
        "precision": precision_score(mobility_true, mobility_pred, pos_label=1, zero_division=0),
        "recall": recall_score(mobility_true, mobility_pred, pos_label=1, zero_division=0),
        "f1": f1_score(mobility_true, mobility_pred, pos_label=1, zero_division=0),
    }
    mob_metrics.append(mob_row)
    print(
        f"  Mobility: Acc={mob_row['acc']:.4f}  P={mob_row['precision']:.4f}  "
        f"R={mob_row['recall']:.4f}  F1={mob_row['f1']:.4f}"
    )

    type_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    type_true, type_pred = [], []
    for train_index, test_index in type_cv.split(X_type, y_type):
        classifier = XGBClassifier(
            objective="multi:softprob",
            num_class=3,
            eval_metric="mlogloss",
            random_state=seed,
            **BEST_PARAMS_TYPE,
        )
        classifier.fit(X_type.iloc[train_index], y_type.iloc[train_index])
        type_true.extend(y_type.iloc[test_index].to_numpy())
        type_pred.extend(classifier.predict(X_type.iloc[test_index]))

    type_true = np.asarray(type_true)
    type_pred = np.asarray(type_pred)
    labels = [0, 1, 2]
    precision_per_class = precision_score(
        type_true, type_pred, labels=labels, average=None, zero_division=0
    )
    recall_per_class = recall_score(
        type_true, type_pred, labels=labels, average=None, zero_division=0
    )
    f1_per_class = f1_score(
        type_true, type_pred, labels=labels, average=None, zero_division=0
    )
    accuracy_per_class = [
        accuracy_score(type_true == class_id, type_pred == class_id) for class_id in labels
    ]

    type_row = {"seed": seed, "acc": accuracy_score(type_true, type_pred)}
    for class_id in labels:
        prefix = label_names[class_id]
        type_row[f"{prefix}_accuracy"] = accuracy_per_class[class_id]
        type_row[f"{prefix}_precision"] = precision_per_class[class_id]
        type_row[f"{prefix}_recall"] = recall_per_class[class_id]
        type_row[f"{prefix}_f1"] = f1_per_class[class_id]
    type_row["macro_precision"] = precision_score(
        type_true, type_pred, average="macro", zero_division=0
    )
    type_row["macro_recall"] = recall_score(
        type_true, type_pred, average="macro", zero_division=0
    )
    type_row["macro_f1"] = f1_score(
        type_true, type_pred, average="macro", zero_division=0
    )
    type_metrics.append(type_row)
    print(
        f"  Carrier-type classifier: Acc={type_row['acc']:.4f}  "
        f"macro P={type_row['macro_precision']:.4f}  "
        f"R={type_row['macro_recall']:.4f}  F1={type_row['macro_f1']:.4f}"
    )

print("\n========== Cross-validation complete ==========")

### 1.2 Mean and standard deviation across seeds

In [ ]:
require_verified_env()

df_mob = pd.DataFrame(mob_metrics)
df_type = pd.DataFrame(type_metrics)


def mean_std(series):
    return f"{series.mean():.4f} +/- {series.std():.4f}"


print("Mobility classifier (10 seeds; mean +/- sample SD)")
for metric, column in [
    ("Accuracy", "acc"),
    ("Precision", "precision"),
    ("Recall", "recall"),
    ("F1", "f1"),
]:
    print(f"  {metric:<10} {mean_std(df_mob[column])}")

print("\nCarrier-type classifier (10 seeds; mean +/- sample SD)")
print(f"  Overall accuracy  {mean_std(df_type['acc'])}")
for display_name, prefix in [("P-type", "P"), ("N-type", "N"), ("Ambipolar", "Amb")]:
    print(f"  {display_name}")
    for metric in ["accuracy", "precision", "recall", "f1"]:
        print(f"    {metric:<10} {mean_std(df_type[f'{prefix}_{metric}'])}")
print("  Macro average")
for metric in ["precision", "recall", "f1"]:
    print(f"    {metric:<10} {mean_std(df_type[f'macro_{metric}'])}")

OUTPUT_DIR.mkdir(exist_ok=True)
df_mob.to_csv(OUTPUT_DIR / "cv_mobility_per_seed.csv", index=False)
df_type.to_csv(OUTPUT_DIR / "cv_carrier_type_per_seed.csv", index=False)
print(f"\nPer-seed metrics saved in: {OUTPUT_DIR}")

### 1.3 Per-seed metrics

In [ ]:
require_verified_env()

print("Mobility classifier:")
print(df_mob.to_string(index=False))
print("\nCarrier-type classifier:")
print(df_type.to_string(index=False))

---
## Part 2. Reproduce and verify the released models

### 2.1 Reproduce the released models

In [ ]:
require_verified_env()

final_mob = XGBClassifier(
    random_state=RELEASE_MODEL_RANDOM_STATE,
    eval_metric="logloss",
    **BEST_PARAMS_MOBILITY,  # Hyperparameters selected by Bayesian optimization.
)
final_mob.fit(X_mob, y_mob)

final_type = XGBClassifier(
    objective="multi:softprob",
    random_state=RELEASE_MODEL_RANDOM_STATE,
    num_class=3,
    eval_metric="mlogloss",
    **BEST_PARAMS_TYPE,  # Hyperparameters selected by Bayesian optimization.
)
final_type.fit(X_type, y_type)

print("Mobility classifier reproduced.")
print(f"  training matrix: {X_mob.shape}; trees: {final_mob.n_estimators}")
print("Carrier-type classifier reproduced.")
print(f"  training matrix: {X_type.shape}; trees: {final_type.n_estimators}")
print("Model verification pending; no model files have been written.")

### 2.2 Verify against the released models and save verified copies



In [ ]:
require_verified_env()

import io
import os

released_mob_path = MODEL_DIR / "mobility_classifier.pkl"
released_type_path = MODEL_DIR / "carrier_type_classifier.pkl"
released_mob = joblib.load(released_mob_path)
released_type = joblib.load(released_type_path)


def serialize_model(model):
    buffer = io.BytesIO()
    joblib.dump(model, buffer)
    return buffer.getvalue()


def verify_model(name, generated, released, released_path, X):
    generated_bytes = serialize_model(generated)
    released_bytes = released_path.read_bytes()
    checks = {
        "predictions": np.array_equal(generated.predict(X), released.predict(X)),
        "probabilities": np.array_equal(
            generated.predict_proba(X), released.predict_proba(X)
        ),
        "booster bytes": (
            generated.get_booster().save_raw(raw_format="ubj")
            == released.get_booster().save_raw(raw_format="ubj")
        ),
        ".pkl bytes": generated_bytes == released_bytes,
    }
    generated_hash = hashlib.sha256(generated_bytes).hexdigest()

    print(f"{name}:")
    for check_name, passed in checks.items():
        print(f"  {check_name + ' identical:':<27} {passed}")
    print(f"  SHA-256: {generated_hash}")

    failed_checks = [check_name for check_name, passed in checks.items() if not passed]
    if failed_checks:
        raise RuntimeError(
            f"{name} does not reproduce the released artifact: {failed_checks}. "
            "Possible causes include a version, data, metadata, or code mismatch. "
        )
    return generated_bytes


verified_mob_bytes = verify_model(
    "Mobility classifier",
    final_mob,
    released_mob,
    released_mob_path,
    X_mob,
)
verified_type_bytes = verify_model(
    "Carrier-type classifier",
    final_type,
    released_type,
    released_type_path,
    X_type,
)

verified_outputs = {
    MODEL_OUTPUT_DIR / "mobility_classifier.pkl": verified_mob_bytes,
    MODEL_OUTPUT_DIR / "carrier_type_classifier.pkl": verified_type_bytes,
}
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for output_path, model_bytes in verified_outputs.items():
    temporary_path = output_path.with_suffix(output_path.suffix + ".tmp")
    try:
        temporary_path.write_bytes(model_bytes)
        os.replace(temporary_path, output_path)
    finally:
        if temporary_path.exists():
            temporary_path.unlink()

print("\nBoth released classifiers were verified exactly.")
print(f"Verified model files saved in: {MODEL_OUTPUT_DIR}")

---
## Outputs

The notebook writes the two per-seed CV metric tables to `model_training_outputs/`.The verified files are written atomically to `model_training_outputs/verified_models/`. 